In [5]:
from sympy import symbols, Matrix, kronecker_product, solve, S

# Step 1: Define base single-qubit states as SymPy Matrices
q0 = Matrix([1, 0])  # |0>
q1 = Matrix([0, 1])  # |1>

# Step 2: Define Alice's unknown 1-qubit input state parameters
alpha, beta = symbols('alpha beta')
input_state = alpha * q0 + beta * q1

# Step 3: Set up the 2-qubit Channel Template with 4 unknown variables
c1, c2, c3, c4 = symbols('c1 c2 c3 c4')
basis_2qubits = [
    kronecker_product(q0, q0),  # |00>
    kronecker_product(q0, q1),  # |01>
    kronecker_product(q1, q0),  # |10>
    kronecker_product(q1, q1)   # |11>
]
unknown_channel = c1*basis_2qubits[0] + c2*basis_2qubits[1] + c3*basis_2qubits[2] + c4*basis_2qubits[3]

# Step 4: Combine into a joint 3-qubit state space vector
total_system = kronecker_product(input_state, unknown_channel)

# Step 5: Define Alice's Bell measurement state vector
bell_plus = (1 / symbols('sqrt(2)')) * (kronecker_product(q0, q0) + kronecker_product(q1, q1))

# Step 6: Apply the projection loop to isolate Bob's leftover state vector
# We perform a partial inner product over Alice's 2 qubits (dimension 4)
bob_leftover = Matrix([0, 0])
for j in range(2):
    val = 0
    for i in range(4):
        val += bell_plus[i] * total_system[i * 2 + j]
    bob_leftover[j] = val

# Step 7: Match Bob's state to Alice's original target state
target_state = Matrix([alpha, beta])

# Step 8: Isolate equations by comparing coefficients for alpha and beta
# This forces SymPy to solve for generic states without parameterizing
eq1_alpha = bob_leftover[0].coeff(alpha) - target_state[0].coeff(alpha)
eq1_beta  = bob_leftover[0].coeff(beta)  - target_state[0].coeff(beta)
eq2_alpha = bob_leftover[1].coeff(alpha) - target_state[1].coeff(alpha)
eq2_beta  = bob_leftover[1].coeff(beta)  - target_state[1].coeff(beta)

# Step 9: Solve the linear constraint system
solution = solve([eq1_alpha, eq1_beta, eq2_alpha, eq2_beta], (c1, c2, c3, c4))

# Step 10: Print the discovered channel constants
print("=== Channel Discovery Complete [Normalization Required] ===")
print(solution)

=== Channel Discovery Complete [Normalization Required] ===
{c1: sqrt(2), c2: 0, c3: 0, c4: sqrt(2)}
